# Category Recommendation and Market Basket Analysis in Olist’s E-commerce

In [62]:
# Load the libraries

from pathlib import Path

import pandas as pd
import numpy as np    

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity   
from IPython.display import display, Markdown, HTML

In [63]:
import warnings 
warnings.filterwarnings('ignore')

## Database

 <p>Olist’s data is divided in multiple datasets. This approach was adopted to facilitate understanding and organization.</p>

 <p>The Olist database consists of 9 files, following an analysis, the following were selected:</p>
     
* olist_orders_dataset
* olist_customers_dataset
* olist_order_items_dataset
* olist_products_dataset
  
<p>Combining these files makes it possible to identify which categories appear in each order, as well as to track purchases made by individual customers over time.</p>

  https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce


In [64]:
# Load dataset into a DataFrame
orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
items = pd.read_csv("olist_order_items_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")

# Print number of rows and columns in dataset
print("orders:", orders.shape)  
print("customers:",customers.shape) 
print("items:", items.shape) 
print("products:", products.shape) 

orders: (99441, 8)
customers: (99441, 5)
items: (112650, 7)
products: (32951, 9)


In [65]:
# Reading of the files with selected columns

orders = pd.read_csv("olist_orders_dataset.csv",
    usecols=["order_id", "customer_id", "order_status", "order_purchase_timestamp"],)

customers = pd.read_csv("olist_customers_dataset.csv",
    usecols=["customer_id", "customer_unique_id", "customer_state"],)


items = pd.read_csv("olist_order_items_dataset.csv",
    usecols=["order_id", "order_item_id", "product_id"],)


products = pd.read_csv("olist_products_dataset.csv",
    usecols=["product_id", "product_category_name", "product_photos_qty",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",],)


display (HTML("<strong>Orders</strong>"))
display(order.head())

display (HTML("<strong>Customers</strong>"))
display(customer.head())

display (HTML("<strong>Items</strong>"))
display(items.head())

display (HTML("<strong>Products</strong>"))
display(products.head())


,order_id,customer_id,order_status,order_purchase_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39


,customer_id,customer_unique_id,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,SP


,order_id,order_item_id,product_id
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089


,product_id,product_category_name,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,4.0,625.0,20.0,17.0,13.0


### Data Preparation

In [68]:
order_orig = orders.copy()

# 1. Check that the keys are populated and unique

assert orders["order_id"].notna().all() and orders["order_id"].is_unique
assert customers["customer_id"].notna().all() and customers["customer_id"].is_unique
assert products["product_id"].notna().all() and products["product_id"].is_unique

# Check the combination of order and item position

assert not items.duplicated(["order_id", "order_item_id"]).any()

# 2. Purchase date conversion

orders["order_purchase_timestamp"] = pd.to_datetime( orders["order_purchase_timestamp"], errors="coerce")

# 3. Selects delivered orders with a valid date

delivered = order.loc[orders["order_status"] == "delivered"].copy()

delivered_dated = delivered.dropna(subset=["order_purchase_timestamp"]).copy()

# 4. Consolidates information from the selected orders

purchases = (delivered_dated.merge(customers,
        on="customer_id",
        how="left",
        validate="many_to_one").merge(items,
        on="order_id",
        how="left",
        validate="one_to_many").merge(products,
        on="product_id",
        how="left",
        validate="many_to_one"))

# 5. Standardize category and remove rows without a category

purchases ["product_category_name"] = (purchases["product_category_name"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA))

with_category = purchases.dropna(subset=["product_category_name"]).copy()

# 6. Check how many orders remained at each stage
audit = pd.Series({
    "Origin orders": orders["order_id"].nunique(),
    "Orders delivered": delivered["order_id"].nunique(),
    "Orders delivered with a valid date": delivered_dated["order_id"].nunique(),
    "Orders with at least one category": with_category["order_id"].nunique(),}, name="Orders").to_frame()

audit["Removed orders"] = (-audit["Orders"].diff().fillna(0)).astype(int)

display(audit)

,Orders,Removed orders
Origin orders,99441,0
Orders delivered,96478,2963
Orders delivered with a valid date,96478,0
Orders with at least one category,95146,1332


In [74]:
#Count of each category per order - once

category_records = with_category.drop_duplicates(subset=["order_id", "product_category_name"]).copy()

display(category_records.head())

,order_id,customer_id,order_status,order_purchase_timestamp,customer_unique_id,customer_state,order_item_id,product_id,product_category_name,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,7c396fd4830fd04220f754e42b4e5bff,SP,1,87285b34884572647811a353c7ac498a,utilidades_domesticas,4.0,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,af07308b275d755c9edb36a90c618231,BA,1,595fac2a385ac33a80bd5114aec74eb8,perfumaria,1.0,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,3a653a41f6f9fc3d2a113cf8398680e8,GO,1,aa4383b373c6aca5d8797843e5594415,automotivo,1.0,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,7c142cf63193a1473d2e66489a9ae977,RN,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,3.0,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,72632f0f9dd73dfee390c9b22eb56dd6,SP,1,65266b2da20d04dbe00c5c2d3bb7859e,papelaria,4.0,250.0,51.0,15.0,15.0


In [78]:
#Category grouped by order identifier

baskets = (category_records.groupby("order_id")["product_category_name"].agg(set))

display(baskets.head())

order_id
00010242fe8c5a6d1ba2dd792cb16214            {cool_stuff}
00018f77f2f0320c557190d7a144bdd3              {pet_shop}
000229ec398224ef6ca0657da4fc703e      {moveis_decoracao}
00024acbcdf0a6daa1e931b038114c75            {perfumaria}
00042b26cf59d7ce69dfabb4e55b4fd9    {ferramentas_jardim}
Name: product_category_name, dtype: object

In [76]:
#checking basket sizes

basket_size = baskets.map(len)

print("Total orders:", len(baskets))

print("Orders with two or more categories:", (basket_size >= 2).sum())

print("Percentage with two or more categories:", f"{(basket_size >= 2).mean():.2%}")

display(basket_size.value_counts()
    .sort_index()
    .rename_axis("Categories by order")
    .to_frame("Number of orders"))

Total orders: 95146
Orders with two or more categories: 723
Percentage with two or more categories: 0.76%


,Number of orders
Categories by order,
1,94423
2,708
3,15


In [77]:
# Purchase History by Customer.

history = category_records.dropna(subset=["customer_unique_id"]).copy()

orders_per_customer = (history.groupby("customer_unique_id")["order_id"].nunique())

print("Identified customers:", orders_per_customer.size)

print("Customers with two or more orders:", (orders_per_customer >= 2).sum())

print("Percentage of customers with two or more orders:",f"{(orders_per_customer >= 2).mean():.2%}")

Identified customers: 92098
Customers with two or more orders: 2742
Percentage of customers with two or more orders: 2.98%


# Reference